# Electricity Price Forecasting
### Predicting Hourly Day-Ahead Spot Prices (€/MWh) for Spain

**Project 18 of 50 — Time Series Forecasting Portfolio**

## Project Overview

Electricity spot prices are among the most volatile and difficult-to-forecast financial time series in existence. Unlike load (which follows predictable human schedules), prices are driven by the **merit order** — the intersection of steeply rising supply curves and relatively inelastic demand.

We forecast **hourly day-ahead electricity spot prices (€/MWh)** for Spain using the ENTSO-E / OMIE dataset (same Kaggle source as the Energy Load notebook).

| Attribute | Value |
|---|---|
| **Dataset** | 
icholasjhana/energy-consumption-generation-prices-and-weather |
| **Target** | price actual (€/MWh) |
| **Date column** | 	ime |
| **Frequency** | Hourly |
| **TS Library** | Darts |
| **Key challenge** | Extreme spikes, regime changes, near-random-walk behaviour |

## Learning Objectives

1. Understand why electricity prices are fundamentally different from load (non-Gaussian, spike-prone, driven by renewable penetration)
2. Handle extreme price spikes in preprocessing without naive clipping that destroys the real signal
3. Explore the relationship between renewable generation and price suppression (merit-order effect)
4. Build appropriate baselines for a high-volatility near-random-walk series
5. Apply FLAML and Darts models, evaluating honestly on a spike-heavy period
6. Discuss why MAPE is a poor metric for price series with near-zero or negative values
7. Identify the limits of purely historical data for price forecasting

## Problem Statement

Forecast the next **24 hours** of hourly electricity prices (€/MWh) for the Spanish day-ahead market.

Unlike load forecasting (170h horizon), electricity prices are typically forecast only 24–48 hours ahead because the underlying drivers (plant availability, fuel prices, renewable output) change rapidly.

## Why This Project Matters

Energy traders, industrial consumers, and storage operators all need accurate price forecasts to:
- **Bid optimally** in the day-ahead auction (closes 12:00 CET the day before)
- **Schedule flexible loads** (electrolysers, pumped hydro, EV charging) to low-price hours
- **Hedge financial exposure** on forward contracts

A 10% improvement in price RMSE can translate into millions of euros of annual value for large industrial consumers.

## Dataset Overview

Same source as the Energy Load notebook: 
icholasjhana/energy-consumption-generation-prices-and-weather.

**Price-relevant columns:**
| Column | Relevance to price |
|---|---|
| price actual | **Target** — hourly OMIE day-ahead price €/MWh |
| price day ahead | Official OMIE forecast price (upper baseline) |
| generation solar | High solar → lower midday prices (merit-order effect) |
| generation wind onshore | High wind → lower prices overall |
| generation fossil gas | Gas is marginal plant → sets price floor |
| 	otal load actual | Higher load → higher prices |

**Price characteristics:**
- Range: often −10 to 200 €/MWh; rare spikes to 3000 €/MWh during scarcity events
- Near-zero prices during high-renewable periods (negative prices possible after 2019)
- Strong daily seasonality but much higher variance than load

## Dataset Source & License

- **Kaggle:** 
icholasjhana/energy-consumption-generation-prices-and-weather
- **Upstream:** OMIE (Spanish electricity market operator) — open data
- **License:** CC BY 4.0
- **Disclaimer:** This notebook is educational only. Electricity trading involves substantial financial risk.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","darts","statsmodels","scipy"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from darts import TimeSeries as DartsSeries
from darts.models import ExponentialSmoothing, AutoARIMA as DartsAutoARIMA, NaiveSeasonal
from darts.metrics import mae as d_mae, rmse as d_rmse

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

pd.set_option("display.max_columns",40)
plt.rcParams["figure.figsize"] = (14,5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT     = "Electricity Price Forecasting"
KAGGLE_SLUG = "nicholasjhana/energy-consumption-generation-prices-and-weather"
TARGET      = "price actual"
TIME_COL    = "time"
FREQ        = "h"
SEASON      = 24   # primary seasonality: daily
HORIZON     = 24   # forecast 24 hours ahead
TEST_SIZE   = HORIZON * 7   # test on a full week (7 × 24h)
VAL_SIZE    = HORIZON * 7
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Target: {TARGET} | Horizon: {HORIZON}h | Test: {TEST_SIZE}h")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or
        os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())

if cred:
    print("Kaggle credentials found.")
else:
    print("="*55)
    print("WARNING: No Kaggle credentials found.")
    print("Visit https://www.kaggle.com/settings → Create New Token")
    print("Place kaggle.json at ~/.kaggle/kaggle.json")
    print("Or set KAGGLE_USERNAME + KAGGLE_KEY environment variables")
    print("="*55)

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data path: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}. Trying CLI...")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print(f"Files: {[f.name for f in csv_files]}")

In [ ]:
# Load the energy dataset
energy_file = next((f for f in csv_files if "energy_dataset" in f.name), csv_files[0])
df_raw = pd.read_csv(energy_file)
df_raw["time"] = pd.to_datetime(df_raw["time"], utc=True).dt.tz_convert("Europe/Madrid").dt.tz_localize(None)
print(f"Shape: {df_raw.shape}")
print(f"Price column stats: {df_raw[TARGET].describe().round(2).to_dict()}")
df_raw[[TIME_COL, TARGET, "price day ahead", "generation solar", "generation wind onshore"]].head(3)

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*50)
miss = df_raw[TARGET].isna().sum()
print(f"Missing in '{TARGET}': {miss} ({miss/len(df_raw)*100:.2f}%)")
dupes = df_raw[TIME_COL].duplicated().sum()
print(f"Duplicate timestamps : {dupes}")
print(f"Price range          : {df_raw[TARGET].min():.2f} to {df_raw[TARGET].max():.2f} EUR/MWh")
print(f"Negative prices      : {(df_raw[TARGET]<0).sum()}")
print(f"Zero prices          : {(df_raw[TARGET]==0).sum()}")
print(f"Extreme spikes (>200): {(df_raw[TARGET]>200).sum()}")

# Year-by-year price stats to detect regime changes
df_raw["year"] = df_raw[TIME_COL].dt.year
print("
Year-by-year price summary (EUR/MWh):")
print(df_raw.groupby("year")[TARGET].agg(["mean","std","min","max"]).round(2).to_string())

## Exploratory Data Analysis

In [ ]:
ts_df = (df_raw[[TIME_COL, TARGET]]
         .drop_duplicates(TIME_COL)
         .sort_values(TIME_COL)
         .rename(columns={TIME_COL:"ds", TARGET:"y"})
         .dropna())

print(f"Series length: {len(ts_df):,} hourly observations")

fig = px.line(ts_df, x="ds", y="y",
              title="Spain Hourly Electricity Price (EUR/MWh) — Full Series",
              labels={"y":"Price (EUR/MWh)", "ds":"Date"})
fig.update_layout(template="plotly_white")
fig.show()

### Price Distribution

Electricity prices are highly non-Gaussian — right-skewed with heavy tails from scarcity spikes. This affects choice of loss function and evaluation metrics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw distribution
axes[0].hist(ts_df["y"], bins=80, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].set_title("Price Distribution (raw)")
axes[0].set_xlabel("EUR/MWh")

# Winsorised distribution (for modelling view)
p01, p99 = ts_df["y"].quantile(0.01), ts_df["y"].quantile(0.99)
winsor = ts_df["y"].clip(p01, p99)
axes[1].hist(winsor, bins=60, color="coral", edgecolor="black", alpha=0.7)
axes[1].set_title(f"Winsorised [p1={p01:.0f}, p99={p99:.0f}]")
axes[1].set_xlabel("EUR/MWh")

# Hour-of-day profile
ts_df["hour"] = ts_df["ds"].dt.hour
ts_df.groupby("hour")["y"].mean().plot(ax=axes[2], marker="o", color="darkgreen")
axes[2].set_title("Average Price by Hour of Day")
axes[2].set_xlabel("Hour"); axes[2].set_ylabel("Mean EUR/MWh")

plt.tight_layout(); plt.show()

In [ ]:
# Merit-order effect: price vs renewable generation
ts_df["year"] = ts_df["ds"].dt.year
df_rn = df_raw[["time","generation solar","generation wind onshore"]].copy()
df_rn.columns = ["ds","solar","wind"]
df_merged = ts_df.merge(df_rn, on="ds", how="left")
df_merged["renewable"] = df_merged["solar"].fillna(0) + df_merged["wind"].fillna(0)

fig = px.scatter(df_merged.sample(5000, random_state=42),
                 x="renewable", y="y", trendline="lowess", opacity=0.3,
                 title="Merit-Order Effect: Renewable Generation vs Price",
                 labels={"renewable":"Solar+Wind (MW)", "y":"Price (EUR/MWh)"})
fig.update_layout(template="plotly_white")
fig.show()
print("Note: higher renewable generation pushes prices DOWN (merit-order effect)")

In [ ]:
# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(14,5))
plot_acf(ts_df["y"].dropna(), lags=48, ax=axes[0], title="ACF — 48 lags (2 days)")
plot_pacf(ts_df["y"].dropna(), lags=48, ax=axes[1], title="PACF — 48 lags (2 days)")
plt.tight_layout(); plt.show()

In [ ]:
# Stationarity
adf = adfuller(ts_df["y"].dropna())
print(f"ADF Statistic: {adf[0]:.4f}  |  p-value: {adf[1]:.4f}")
print("STATIONARY" if adf[1]<0.05 else "NON-STATIONARY — may need differencing")
print()
print("Important: unlike load, price stationarity is regime-dependent.")
print("A 4-year test may show stationarity even when sub-periods show drift.")

## Target Analysis

Key characteristics specific to electricity prices:
1. **Spike asymmetry**: positive spikes (scarcity) are larger than negative spikes
2. **Price floors**: near-zero during high renewable penetration
3. **Regime changes**: structural price level shifts between years (carbon price, new capacity)
4. **Intraday pattern**: weaker and more variable than load

In [ ]:
print("Price statistics:")
print(ts_df["y"].describe().round(2))
print()

# Spike analysis
iqr = ts_df["y"].quantile(0.75) - ts_df["y"].quantile(0.25)
spike_thresh = ts_df["y"].quantile(0.75) + 3*iqr
spikes = ts_df[ts_df["y"] > spike_thresh]
print(f"Spike threshold (Q3 + 3*IQR): {spike_thresh:.1f} EUR/MWh")
print(f"Spike hours: {len(spikes)} ({len(spikes)/len(ts_df)*100:.2f}%)")
print()
print("Top 10 most expensive hours:")
print(ts_df.nlargest(10, "y")[["ds","y"]].to_string(index=False))

## Train / Validation / Test Split

We use the **last 168h (1 week) as test**, previous 168h as validation. Temporal split only — no shuffling.

In [ ]:
n = len(ts_df)
ts_test  = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val   = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} h  {ts_train['ds'].min().date()} → {ts_train['ds'].max().date()}")
print(f"Val:    {len(ts_val):,} h  {ts_val['ds'].min().date()} → {ts_val['ds'].max().date()}")
print(f"Test:   {len(ts_test):,} h  {ts_test['ds'].min().date()} → {ts_test['ds'].max().date()}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("Split is leak-free.")

## Preprocessing

**Critical decision for prices:** We do NOT naively clip extreme spikes. Instead we log-transform the target to reduce spike influence on MSE-based losses, then back-transform for evaluation.

Why log-transform? The spike at €3000 should not dominate the MSE 10,000× more than a typical hour at €30.

In [ ]:
def preprocess_price(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    n_miss = df["y"].isna().sum()
    if n_miss > 0:
        df["y"] = df["y"].interpolate("linear")
        print(f"  Interpolated {n_miss} NaNs")
    # Floor at -100 EUR/MWh (theoretical minimum for negative price periods)
    n_clip = (df["y"] < -100).sum()
    if n_clip > 0:
        df["y"] = df["y"].clip(-100, None)
        print(f"  Clipped {n_clip} extreme negative values")
    return df

ts_train    = preprocess_price(ts_train)
ts_val      = preprocess_price(ts_val)
ts_test     = preprocess_price(ts_test)
ts_trainval = preprocess_price(ts_trainval)

# Log-scale version for MSE-based modelling (add shift so all values > 0)
PRICE_SHIFT = abs(ts_trainval["y"].min()) + 1
ts_train_log = ts_train.copy(); ts_train_log["y"] = np.log1p(ts_train["y"] + PRICE_SHIFT)
ts_val_log   = ts_val.copy();   ts_val_log["y"]   = np.log1p(ts_val["y"]   + PRICE_SHIFT)
print(f"Price shift constant: {PRICE_SHIFT:.2f} EUR/MWh")
print("Preprocessing done.")

## Feature Engineering

Features specific to electricity price forecasting:
- **price lags**: 1h, 2h, 24h (yesterday same hour), 168h (last week same hour)
- **rolling price stats**: 24h mean/std
- **calendar**: hour, DOW — but with less explanatory power than for load
- **renewable proxy lags**: would ideally include solar/wind forecast, but we only have historical values

In [ ]:
def make_price_features(df_in):
    df = df_in.copy()
    for lag in [1, 2, 3, 24, 168]:
        df[f"price_lag_{lag}h"] = df["y"].shift(lag)
    for w in [6, 24]:
        df[f"roll_mean_{w}h"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}h"]  = df["y"].shift(1).rolling(w).std()
    df["hour"]       = df["ds"].dt.hour
    df["dow"]        = df["ds"].dt.dayofweek
    df["month"]      = df["ds"].dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["hour_sin"]   = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]   = np.cos(2*np.pi*df["hour"]/24)
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_price_features(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y","hour","dow","month"]]

train_f = ts_feat.iloc[:len(ts_train)].dropna().copy()
val_f   = ts_feat.iloc[len(ts_train):len(ts_train)+len(ts_val)].dropna().copy()
test_f  = ts_feat.iloc[len(ts_train)+len(ts_val):].dropna().copy()
print(f"Features: {feat_cols}")

## Baseline Approaches

In [ ]:
results = []

def evaluate(actual, pred, name):
    a, p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    # Use sMAPE (symmetric MAPE) to handle near-zero prices
    smape = np.mean(2*np.abs(a-p)/(np.abs(a)+np.abs(p)+1e-9))*100
    print(f"{name:<38s} MAE={mae:7.2f}  RMSE={rmse:7.2f}  sMAPE={smape:5.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"sMAPE":smape})

# Naive: last known price
evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].iloc[-1]), "Naive (last price)")

# Seasonal Naive: same hour yesterday
sn24 = np.tile(ts_trainval["y"].iloc[-SEASON:].values, len(ts_test)//SEASON+1)[:len(ts_test)]
evaluate(ts_test["y"], sn24, "Seasonal Naive (same hour yesterday)")

# Seasonal Naive: same hour last week
sn168 = np.tile(ts_trainval["y"].iloc[-168:].values, len(ts_test)//168+1)[:len(ts_test)]
evaluate(ts_test["y"], sn168, "Seasonal Naive (same hour last week)")

print()
print("Note: sMAPE is used instead of MAPE to handle near-zero electricity prices.")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
print(f"Input: train {X_tr.shape}, val {X_va.shape}")
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(15).to_string())
    lazy_top5 = lm.head(5).index.tolist()
    print(f"Top-5: {lazy_top5}")
except Exception as e:
    print(f"LazyPredict error: {e}"); lazy_top5 = []

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]

flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
           time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(X_te)

print(f"Best model: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## Darts — Dedicated Time-Series Library

**Why Darts for electricity prices?**

Darts provides a unified interface to try multiple model families quickly — essential for a problem where no single model family dominates. We test three approaches with increasing complexity:
1. **NaiveSeasonal (24h)** — same-hour-yesterday, our upper bound for "no intelligence" needed
2. **ExponentialSmoothing** — handles daily seasonality + trend
3. **AutoARIMA** — auto-selects the best ARIMA(p,d,q)(P,D,Q) structure

We also note the theoretical limit of pure lagged-price forecasting: fundamentals like wind ramps, gas prices, and demand shocks are not in the price history alone.

In [ ]:
def to_darts(df):
    s = df.set_index("ds")["y"].asfreq("h").interpolate()
    return DartsSeries.from_series(s)

train_d = to_darts(ts_trainval)
test_d  = to_darts(ts_test)
h = len(ts_test)
darts_preds = {}

# 1. Seasonal Naive 24h
try:
    sn = NaiveSeasonal(K=SEASON)
    sn.fit(train_d); p = sn.predict(h)
    darts_preds["Darts-SeasonalNaive-24h"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-SeasonalNaive-24h")
except Exception as e: print(f"SeasonalNaive failed: {e}")

# 2. ETS
try:
    ets = ExponentialSmoothing(seasonal_periods=SEASON, trend=True, seasonal="add")
    ets.fit(train_d); p = ets.predict(h)
    darts_preds["Darts-ETS"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-ETS")
except Exception as e: print(f"ETS failed: {e}")

# 3. AutoARIMA
try:
    arima = DartsAutoARIMA()
    arima.fit(train_d); p = arima.predict(h)
    darts_preds["Darts-AutoARIMA"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-AutoARIMA")
except Exception as e: print(f"AutoARIMA failed: {e}")

In [ ]:
# Forecast plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black",width=2)))
colors = ["steelblue","darkorange","green","purple"]
for (nm, pred), col in zip(darts_preds.items(), colors):
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred.values().flatten(),
                              name=nm, line=dict(color=col,dash="dash")))
fig.update_layout(title="Electricity Price Forecasts vs Actual",
                  xaxis_title="Date", yaxis_title="Price (EUR/MWh)",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL RESULTS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Model Comparison — RMSE",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white")
fig.show()

## Final Training & Evaluation of Top 3

In [ ]:
for _, row in top3.iterrows():
    print(f"  {row['model']}  RMSE={row['RMSE']:.2f}  MAE={row['MAE']:.2f}  sMAPE={row['sMAPE']:.2f}%")

# Context + forecast overlay
context_n = min(168, len(ts_trainval))
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts_trainval["ds"].iloc[-context_n:],
    y=ts_trainval["y"].iloc[-context_n:],
    name="Train context", line=dict(color="grey",dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual Test", line=dict(color="black",width=2)))
if flaml_pred is not None and len(test_f)>0:
    fig.add_trace(go.Scatter(x=test_f["ds"], y=flaml_pred,
                              name=f"FLAML", line=dict(dash="dot",color="crimson")))
for nm, pred in list(darts_preds.items())[:2]:
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred.values().flatten(),
                              name=nm, line=dict(dash="dash")))
fig.update_layout(title="Top 3 Models — Test Week Price Forecast",
                  template="plotly_white", xaxis_title="Date", yaxis_title="EUR/MWh")
fig.show()

## Error Analysis

In [ ]:
if flaml_pred is not None and len(test_f) > 0:
    actual = test_f["y"].values
    pred   = flaml_pred[:len(actual)]
    errors = actual - pred
    ae     = np.abs(errors)

    print(f"Forecast bias (mean error): {errors.mean():+.2f} EUR/MWh")
    print(f"MAE                        : {ae.mean():.2f} EUR/MWh")
    print(f"Max absolute error         : {ae.max():.2f} EUR/MWh")
    print(f"Hours with >20 EUR/MWh err : {(ae>20).sum()} / {len(ae)}")

    fig, axes = plt.subplots(1, 3, figsize=(18,5))

    axes[0].hist(errors, bins=40, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", lw=2, ls="--")
    axes[0].set_title("Error Distribution (EUR/MWh)")

    axes[1].plot(range(len(errors)), errors, alpha=0.7, color="coral")
    axes[1].axhline(0, color="red", lw=1, ls="--")
    axes[1].set_title("Errors Over 7-Day Horizon")
    axes[1].set_xlabel("Hour ahead")

    # Actual vs predicted scatter
    axes[2].scatter(actual, pred, alpha=0.4, s=12, color="darkblue")
    lo, hi = min(actual.min(), pred.min()), max(actual.max(), pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted"); axes[2].set_xlabel("Actual (EUR/MWh)")
    axes[2].set_ylabel("Predicted (EUR/MWh)")

    plt.tight_layout(); plt.show()

## Interpretation & Insights

### Key findings
1. **Electricity prices are fundamentally harder to forecast than load.** The merit-order mechanism means small changes in renewable output or demand can cause large, non-linear price jumps
2. **FLAML typically finds an ensemble that beats naive baselines** for regular hours but struggles with spike hours
3. **Darts ETS** can track the daily pattern well but underestimates spike magnitude by design (it assumes exponential smoothing structure, not fat tails)
4. **The ENTSO-E official forecast** (column price day ahead) is a strong competitor — it incorporates forward information (planned plant outages, weather forecasts) not available in pure price history

### When models fail
- During **wind droughts** (3+ days of very low wind), prices spike as fossil plants must cover — no pure price-history model can anticipate this
- During **demand surprises** (unexpected cold snaps), load and price co-spike — again requiring weather input
- During **scarcity events** (plant outages), prices can jump to the regulatory cap instantly — fundamentally unpredictable from price alone

## Limitations

1. **No fundamental data used**: Wind power forecasts, temperature, fuel prices, and planned outages are the real drivers of price — using only lagged prices loses most of the signal
2. **No negative price handling**: Post-2019 negative prices (renewable curtailment avoidance) are not explicitly modelled
3. **Single test week**: Markets go through different regimes; one test week may be unrepresentative
4. **Spike loss function**: MSE/MAE both handle spikes poorly — production systems often use quantile regression or Expected Shortfall for risk-aware forecasting
5. **Regulatory changes**: Carbon market prices (EU ETS) dramatically shift the price floor and are not in this dataset

## How to Improve This Project

1. **Add wind & solar generation forecasts** as exogenous regressors in Darts (future-known covariates)
2. **Use quantile regression** (e.g. GradientBoostingRegressor with loss='quantile') to produce 10th–90th percentile forecasts — far more useful than point forecasts for risk management
3. **Separate spike model**: train a classifier to detect spike hours, then a regression model for spike magnitude
4. **Add EU ETS carbon price** as a daily feature — it explains much of the 2017-2018 price level increase
5. **Try pinball loss with FLAML**: configure metric='pinball' to optimise for a specific quantile

## Production Considerations

1. **Intraday updates**: Re-forecast every hour using the latest actual data — prices can move significantly intraday
2. **Market gate closure**: Day-ahead bids close at 12:00 CET; the model must be ready before this deadline with the next 24h forecast
3. **Regulatory compliance**: Energy trading forecasting systems must be auditable in most jurisdictions
4. **Risk management integration**: The forecasting API should expose not just point forecasts but risk metrics (VaR) at the 95%, 99% levels
5. **Data sovereignty**: OMIE price data is free but the terms of use prohibit redistribution — ensure compliance

## Common Mistakes to Avoid

1. **Using price day ahead as an input feature without careful lagging** — this is the "official" forecast for the same delivery hour, which is only known 24h before delivery, not at time of forecasting
2. **Evaluating only on MAPE**: prices near zero produce infinite MAPE. Always also report MAE and RMSE in absolute EUR/MWh
3. **Ignoring regime changes**: fitting a single model to 4 years of price data ignores structural breaks (capacity changes, carbon price jumps). Consider year-by-year cross-validation
4. **Treating forecasting error symmetrically**: underpredicting during high-price hours and overpredicting during low-price hours have very different financial consequences

## Mini Challenge / Exercises

1. **Beat the official forecast**: compute RMSE for the column price day ahead vs price actual. Does your model outperform OMIE's official forecast?
2. **Spike detection**: train a binary classifier to predict whether the next hour's price will exceed 100 EUR/MWh. Report precision/recall — precision matters most (false alarms are costly)
3. **Add temperature**: merge weather_features.csv on timestamp, add Madrid_temp_k as a feature. How much does RMSE change?
4. **Seasonal price regime split**: split by season (winter/summer) and evaluate separately. Which season is harder to forecast?
5. **Backtesting**: implement a rolling monthly backtest across 2018. Report average and worst-month RMSE

## Final Summary & Key Takeaways

### What We Did
- Loaded and validated the ENTSO-E Spain dataset, focusing on price actual
- Explored the merit-order effect (renewable generation → price suppression)
- Identified price spikes and explained their origin in the power system
- Built a log-shifted preprocessing pipeline to reduce spike domination of MSE
- Engineered price-specific lag features (lag-24h, lag-168h, rolling stats)
- Benchmarked with LazyPredict + FLAML AutoML
- Fitted Darts ETS, AutoARIMA, and SeasonalNaive models
- Selected Top 3 by RMSE, used sMAPE (not MAPE) to handle near-zero prices

### Key Takeaways
1. **Electricity prices are qualitatively different from load** — spiky, non-Gaussian, driven by fundamentals not pattern recognition
2. **Same-hour-yesterday (SNaive 24h) is a very competitive baseline** — if you can barely beat it, re-examine your feature set
3. **Evaluation metric matters**: MAPE fails near-zero prices; use sMAPE or RMSE in absolute EUR/MWh
4. **Fundamental data (wind, temperature, outages) is more valuable than more complex models** — garbage in, garbage out still applies
5. **Quantile forecasting** is more actionable than point forecasting for risk-aware energy trading

---
*Educational notebook — part of the 50-project Time Series Forecasting portfolio.*